# M1-T05: Interactive Visualizations (Smart Shelf, Milestone 1)

Interactive companion to the **EDA Report**. Every chart is built from the verified dataset
(`perishable_goods_management.csv`, 100,000 rows × 42 columns) and each chart is followed by a check that its numbers
match the figures quoted in the report (M1-T02, M1-T03 and M1-T04 results).

**How to run:** ensure `perishable_goods_management.csv` is located in `data/raw/` (on Colab, upload it from the Files panel), then *Run All*.
Each chart is also saved as a small standalone HTML file in `reports/figures/interactive/` (loads Plotly from a CDN, so it needs an internet connection to open).

**How to read the charts:** hover for exact values, click a legend entry to hide a series, drag to zoom, double-click to reset.
All patterns are **associations**, not causes.

| # | Chart | Question it answers |
|---|---|---|
| 1 | Spoilage rate by category | Which categories spoil most? |
| 2 | Category × quality grade | Does the grade effect hold inside every category? |
| 3 | Monthly spoilage by category | Is there any trend or seasonality? |
| 4 | Spoilage across groups of four drivers | Which inputs move the spoilage rate? |
| 5 | Spoilage vs waste by category | Is waste the same thing as spoilage? |
| 6 | Markdown depth and spoilage | How deep are discounts, and do marked-down batches spoil more? |
| 7 | Profit distribution by category | Where do the losses sit? |
| 8 | Heatmap category × region | Does region matter? |
| 9 | Correlation of inputs with spoilage | Which inputs are linearly related to the target? |

## 1. Load and verify the data

In [1]:
import hashlib, sys
from pathlib import Path

# Resolve project root (notebooks/ is 1 level below project root)
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import RAW_DATASET_PATH, FIGURES_INTERACTIVE_DIR

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

px.defaults.template = "plotly_white"

DATA_PATH = RAW_DATASET_PATH
EXPECTED_SHA = "de94302b867c9debedfd45c431306623fdfc038f5ed8ca17736339b4460a6674"   # dataset verified in M1-T01
OUT_DIR = FIGURES_INTERACTIVE_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)

sha = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
print("-> matches the dataset verified in M1-T01" if sha == EXPECTED_SHA
      else "-> WARNING: this file is NOT the one verified in M1-T01. Check it before trusting the charts.")

df = pd.read_csv(DATA_PATH, parse_dates=["transaction_date", "expiration_date"])
assert df.shape == (100_000, 42), "Unexpected shape"
assert df.isnull().sum().sum() == 0, "Unexpected missing values"

overall = df["was_spoiled"].mean()
assert int(df["was_spoiled"].sum()) == 19_442 and round(overall * 100, 1) == 19.4   # EDA report, section 4
print(f"{len(df):,} rows | overall spoilage rate {overall:.1%}")


def show_and_save(fig, name):
    """Show the chart in the notebook and save a small standalone HTML copy."""
    fig.write_html(OUT_DIR / f"{name}.html", include_plotlyjs="cdn")
    fig.show()


-> matches the dataset verified in M1-T01


100,000 rows | overall spoilage rate 19.4%


## 2. Spoilage rate by category

The overall rate is 19.4 %. Ready_to_Eat is highest (22.1 %) and Frozen_Meals lowest (16.9 %). The spread is small (Cramér's V = 0.039).

In [2]:
by_cat = (df.groupby("category")
            .agg(spoilage_rate=("was_spoiled", "mean"),
                 avg_waste_pct=("waste_pct", "mean"),
                 batches=("record_id", "size"))
            .reset_index()
            .sort_values("spoilage_rate"))
cat_order = by_cat["category"].tolist()

# checks against the EDA report
rate = by_cat.set_index("category")["spoilage_rate"]
assert round(rate["Frozen_Meals"], 3) == 0.169 and round(rate["Ready_to_Eat"], 3) == 0.221

fig = px.bar(by_cat, x="spoilage_rate", y="category", orientation="h",
             hover_data={"spoilage_rate": ":.1%", "batches": ":,", "category": False},
             labels={"spoilage_rate": "Spoilage rate", "category": ""},
             title="Spoilage rate by category (dashed line = overall rate)")
fig.add_vline(x=overall, line_dash="dash", line_color="black",
              annotation_text=f"overall {overall:.1%}", annotation_position="top")
fig.update_xaxes(tickformat=".0%", rangemode="tozero")
show_and_save(fig, "01_spoilage_by_category")

## 3. Quality grade inside every category

Grade is the clearest categorical signal (Cramér's V = 0.091): the rate rises from A to B to C in **every** category.
Overall: A 15.3 %, B 20.6 %, C 25.9 %.

In [3]:
grade_cat = (df.groupby(["category", "quality_grade"])["was_spoiled"].mean().reset_index())

# checks: A < B < C inside every category, and the overall grade rates quoted in the report
piv = grade_cat.pivot(index="category", columns="quality_grade", values="was_spoiled")
assert (piv["A"] < piv["B"]).all() and (piv["B"] < piv["C"]).all()
overall_grade = df.groupby("quality_grade")["was_spoiled"].mean().round(3)
assert overall_grade.to_dict() == {"A": 0.153, "B": 0.206, "C": 0.259}

fig = px.bar(grade_cat, x="category", y="was_spoiled", color="quality_grade", barmode="group",
             category_orders={"quality_grade": ["A", "B", "C"], "category": cat_order},
             color_discrete_map={"A": "#2e7d32", "B": "#f9a825", "C": "#c62828"},
             labels={"was_spoiled": "Spoilage rate", "quality_grade": "Quality grade", "category": ""},
             title="Spoilage rate by category and quality grade")
fig.update_yaxes(tickformat=".0%")
show_and_save(fig, "02_category_by_grade")

## 4. Monthly spoilage rate by category

No trend and no seasonality: the overall monthly rate stays between 18.3 % and 20.6 % from 2023-01 to 2024-12.
Individual category lines are noisier because each month holds fewer batches per category. Click a category in the legend to hide it.

In [4]:
monthly = (df.assign(month=df["transaction_date"].dt.to_period("M").dt.to_timestamp())
             .groupby(["month", "category"])["was_spoiled"].agg(["mean", "size"]).reset_index()
             .rename(columns={"mean": "spoilage_rate", "size": "batches"}))

# check: overall monthly rate range quoted in the report
overall_monthly = df.groupby(df["transaction_date"].dt.to_period("M"))["was_spoiled"].mean()
assert round(overall_monthly.min() * 100, 1) == 18.3 and round(overall_monthly.max() * 100, 1) == 20.6

fig = px.line(monthly, x="month", y="spoilage_rate", color="category",
              hover_data={"batches": ":,", "spoilage_rate": ":.1%"},
              labels={"spoilage_rate": "Spoilage rate", "month": ""},
              title="Monthly spoilage rate by category (click a name in the legend to hide it)")
fig.update_yaxes(tickformat=".0%", rangemode="tozero")
show_and_save(fig, "03_monthly_by_category")

## 5. What moves the spoilage rate? Four drivers, split into groups

Batches are split into up to 10 equal-sized groups (group 1 = lowest values, last group = highest) and the spoilage rate is computed per group. `handling_score` and `packaging_score` take few distinct values, so they end up with 6 and 5 groups.
The effects are small but consistent in direction: the rate falls from about 24 % to 14 % as `packaging_score` rises, from about 22 % to 16 % as `handling_score` rises,
and rises from about 18 % to 23 % as `temp_deviation` rises. The share of shelf life left is the right way to use days-to-expiry (the raw `days_until_expiry` has r = −0.004).

In [5]:
drivers = {
    "Share of shelf life left": (df["days_until_expiry"] / df["shelf_life_days"]).clip(0, 1),
    "Temperature deviation": df["temp_deviation"],
    "Handling score": df["handling_score"],
    "Packaging score": df["packaging_score"],
}

rows = []
for name, values in drivers.items():
    groups = pd.qcut(values, 10, duplicates="drop")
    g = df.groupby(groups, observed=True)["was_spoiled"].agg(["mean", "size"])
    for i, (_, r) in enumerate(g.iterrows(), start=1):
        rows.append({"driver": name, "decile": i, "spoilage_rate": r["mean"], "batches": int(r["size"])})
deciles = pd.DataFrame(rows)

# checks: end points quoted in the report
def ends(name):
    s = deciles[deciles["driver"] == name].sort_values("decile")["spoilage_rate"]
    return round(s.iloc[0] * 100), round(s.iloc[-1] * 100)
assert ends("Packaging score") == (24, 14) and ends("Handling score") == (22, 16)
assert ends("Temperature deviation") == (18, 23) and ends("Share of shelf life left") == (22, 18)

fig = px.line(deciles, x="decile", y="spoilage_rate", facet_col="driver", facet_col_wrap=2, markers=True,
              hover_data={"batches": ":,", "spoilage_rate": ":.1%"},
              labels={"spoilage_rate": "Spoilage rate", "decile": "Group (1 = lowest values)"},
              title="Spoilage rate across groups of four drivers")
fig.update_yaxes(tickformat=".0%")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
show_and_save(fig, "04_driver_deciles")

## 6. Spoilage is not the same as waste

Frozen_Meals has the **lowest** spoilage rate but the **highest** average `waste_pct` (50.0 %), and Pharmaceuticals is second on waste (37.5 %) with an above-average spoilage rate.
Waste is driven mainly by shelf life (r = +0.31), storage temperature (−0.27) and demand (−0.23). Bubble size = number of batches.

In [6]:
# checks against the EDA report
waste = by_cat.set_index("category")["avg_waste_pct"]
assert round(waste["Frozen_Meals"], 1) == 50.0 and round(waste["Pharmaceuticals"], 1) == 37.5

fig = px.scatter(by_cat, x="spoilage_rate", y="avg_waste_pct", size="batches", text="category",
                 hover_name="category", hover_data={"spoilage_rate": ":.1%", "avg_waste_pct": ":.1f", "batches": ":,", "category": False},
                 labels={"spoilage_rate": "Spoilage rate", "avg_waste_pct": "Average waste_pct (%)"},
                 title="Spoilage rate vs average waste by category")
fig.update_traces(textposition="top center")
fig.update_xaxes(tickformat=".0%")
show_and_save(fig, "05_spoilage_vs_waste")

## 7. Markdown depth and spoilage

`markdown_applied` is exactly the indicator `discount_pct > 0` (M1-T02: 0 contradictory rows). 30,651 batches were marked down, with discounts from 10 % to 75 %.
60.0 % of markdowns are 30 % or less, and 21.0 % are above 50 %. Marked-down batches spoil more often than the others.
This is consistent with discounts being given to batches that already look risky; it does **not** show that discounts cause spoilage (to be tested in Track 3B).

In [7]:
md_rows = df[df["markdown_applied"] == 1].copy()
labels = ["10-20%", "20-30%", "30-40%", "40-50%", "50-60%", "60-75%"]
md_rows["discount band"] = pd.cut(md_rows["discount_pct"], bins=[0.0999, 0.20, 0.30, 0.40, 0.50, 0.60, 0.75], labels=labels)

depth = (md_rows.groupby("discount band", observed=True)["was_spoiled"]
                .agg(spoilage_rate="mean", batches="size").reset_index())
base_rate = df.loc[df["markdown_applied"] == 0, "was_spoiled"].mean()

# checks against M1-T02
assert depth["batches"].tolist() == [9248, 9155, 2274, 3531, 3185, 3258]
assert len(md_rows) == 30_651 and md_rows["discount_pct"].min() == 0.10 and md_rows["discount_pct"].max() == 0.75
assert ((df["markdown_applied"] == 1) & (df["discount_pct"] == 0)).sum() == 0

fig = px.bar(depth, x="discount band", y="spoilage_rate",
             hover_data={"batches": ":,", "spoilage_rate": ":.1%"},
             labels={"spoilage_rate": "Spoilage rate", "discount band": "Discount depth (marked-down batches only)"},
             title="Spoilage rate by markdown depth (dashed line = batches with no markdown)")
fig.add_hline(y=base_rate, line_dash="dash", line_color="black",
              annotation_text=f"no markdown {base_rate:.1%}", annotation_position="top left")
fig.update_yaxes(tickformat=".0%", rangemode="tozero")
show_and_save(fig, "06_markdown_depth")

## 8. Profit distribution by category

28.7 % of batches have negative profit. The IQR rule flags 15.8 % of profits as outliers; they are **kept** because they are real total-loss batches.
The 100 worst batches (lowest profit −169,495) are all Pharmaceuticals. The boxes below show the quartiles and the whiskers (largest and smallest values inside 1.5 × IQR);
outlier points are not drawn so that the file stays small.

In [8]:
def box_stats(s):
    q1, med, q3 = s.quantile([0.25, 0.5, 0.75])
    iqr = q3 - q1
    return pd.Series({"q1": q1, "median": med, "q3": q3, "mean": s.mean(),
                      "lower": s[s >= q1 - 1.5 * iqr].min(), "upper": s[s <= q3 + 1.5 * iqr].max()})

profit_stats = df.groupby("category")["profit"].apply(box_stats).unstack().loc[cat_order]

# checks against the EDA report
assert round((df["profit"] < 0).mean() * 100, 1) == 28.7
assert df.nsmallest(100, "profit")["category"].eq("Pharmaceuticals").all() and df["profit"].min() == -169_495.0

fig = go.Figure(go.Box(x=profit_stats.index.tolist(),
                       q1=profit_stats["q1"].tolist(), median=profit_stats["median"].tolist(),
                       q3=profit_stats["q3"].tolist(), mean=profit_stats["mean"].tolist(),
                       lowerfence=profit_stats["lower"].tolist(), upperfence=profit_stats["upper"].tolist(),
                       boxmean=True, name="profit"))
fig.update_layout(title="Profit per batch by category (box = quartiles, whiskers = 1.5 × IQR, diamond = mean)",
                  yaxis_title="Profit", showlegend=False)
show_and_save(fig, "07_profit_by_category")

## 9. Category × region

Region barely changes the spoilage rate in any category (region: Cramér's V = 0.007, p = 0.36). The map is nearly uniform within each row.

In [9]:
heat = df.pivot_table(index="category", columns="region", values="was_spoiled", aggfunc="mean").loc[cat_order[::-1]]

# check: region is not a meaningful driver (report: V = 0.007)
from scipy import stats
table = pd.crosstab(df["region"], df["was_spoiled"])
chi2 = stats.chi2_contingency(table)[0]
assert round(np.sqrt(chi2 / (table.values.sum() * (min(table.shape) - 1))), 3) == 0.007

fig = px.imshow(heat, text_auto=".1%", aspect="auto", color_continuous_scale="YlOrRd",
                labels={"color": "Spoilage rate", "x": "Region", "y": ""},
                title="Spoilage rate: category × region")
show_and_save(fig, "08_category_region_heatmap")

## 10. Which inputs are linearly related to spoilage?

Only usable **inputs** are shown. `spoilage_risk` and the seven outcome columns (`units_sold`, `units_wasted`, `waste_pct`, `waste_cost`, `revenue`, `profit`, `profit_margin_pct`)
are excluded because they come from the same process as the target or happen after it (leakage rule from M1-T01). All correlations are weak (|r| < 0.1).
`quality_grade` is categorical and is covered in section 3.
`markdown_applied`, `discount_pct` and `selling_price` are discount treatments: they are shown here for completeness but are excluded from the Track 3A classifier.

In [10]:
excluded = ["spoilage_risk", "units_sold", "units_wasted", "waste_pct", "waste_cost",
            "revenue", "profit", "profit_margin_pct"]
id_like = [c for c in df.columns if c.endswith("_id")]
inputs = [c for c in df.select_dtypes("number").columns if c not in excluded + id_like + ["was_spoiled"]]

corr = df[inputs + ["was_spoiled"]].corr()["was_spoiled"].drop("was_spoiled").sort_values()
corr_df = corr.rename("pearson_r").reset_index().rename(columns={"index": "input"})
corr_df["role"] = np.where(corr_df["input"].isin(["markdown_applied", "discount_pct", "selling_price"]),
                           "discount treatment", "input")

# checks against the EDA report
assert round(corr["packaging_score"], 3) == -0.088 and round(corr["markdown_applied"], 3) == 0.058
assert round(corr["handling_score"], 3) == -0.056 and round(corr["discount_pct"], 3) == 0.047
assert round(corr["temp_deviation"], 3) == 0.043 and corr.abs().max() < 0.1

fig = px.bar(corr_df, x="pearson_r", y="input", orientation="h", color="role",
             color_discrete_map={"input": "#1565c0", "discount treatment": "#ef6c00"},
             hover_data={"pearson_r": ":.3f"},
             labels={"pearson_r": "Pearson r with was_spoiled", "input": ""},
             title="Correlation of each input with was_spoiled")
fig.update_layout(height=700)
show_and_save(fig, "09_input_correlations")

## 11. Summary and files

- Overall spoilage rate **19.4 %** (19,442 of 100,000 batches); every single input is only weakly linked to it, so expect modest predictive power from Track 3A.
- Clearest signals: **quality grade**, **packaging score**, **handling score**, **temperature deviation** and **share of shelf life left**. No effect from region, promotion, month or weekday.
- **Waste is not spoilage**: the highest-waste categories (Frozen_Meals, Pharmaceuticals) are not the highest-spoilage ones.
- Marked-down batches spoil more, which is a hypothesis for the Track 3B confounding check and not a causal result.

In [11]:
print("Saved interactive charts:")
for p in sorted(OUT_DIR.glob("*.html")):
    print(f"  {p.name:40s} {p.stat().st_size / 1024:8.1f} KB")

Saved interactive charts:
  01_spoilage_by_category.html                  8.7 KB
  02_category_by_grade.html                     9.7 KB
  03_monthly_by_category.html                  23.7 KB
  04_driver_deciles.html                       11.6 KB
  05_spoilage_vs_waste.html                     9.0 KB
  06_markdown_depth.html                        8.7 KB
  07_profit_by_category.html                    8.4 KB
  08_category_region_heatmap.html               9.0 KB
  09_input_correlations.html                    9.2 KB
  interactive_monthly_by_category.html       4678.0 KB
  interactive_profit_distribution.html       6839.1 KB
  interactive_spoilage_by_category.html      4664.2 KB
